# Recommender comparison on validation

Run 01_data and 02_popularity in validation mode before running this notebook.
On Colab, push the evaluation branch so the remote kernel can import movierec.evaluation.

All methods start with the same user sample and a catalog of films with at least 10
train ratings. Recall and NDCG use users with at least one validation rating of 4 stars
or more. Recall includes all their validation positives in the denominator. Novelty,
genre diversity, long-tail share and catalog coverage use the full sample. Popularity and
the long-tail boundary are calculated from train.

In [ ]:
import importlib
import os
import subprocess
import sys
import types

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None

if drive is not None:
    from IPython import get_ipython

    shell = get_ipython()
    BRANCH = "feat/evaluation"
    REPO_URL = "https://github.com/Cxsee/AI-in-Industry-project.git"
    EXTRAS = ""
    DRIVE_DIR = "/content/drive/MyDrive/movierec"
    REPO_DIR = "/content/AI-in-Industry-project-evaluation"

    drive.mount("/content/drive")
    os.environ["MOVIEREC_DATA_DIR"] = f"{DRIVE_DIR}/data"
    os.environ["MOVIEREC_RESULTS_DIR"] = f"{DRIVE_DIR}/results"
    os.makedirs(os.environ["MOVIEREC_DATA_DIR"], exist_ok=True)
    os.makedirs(os.environ["MOVIEREC_RESULTS_DIR"], exist_ok=True)

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)

    shell.run_line_magic("cd", REPO_DIR)
    subprocess.run(["git", "pull", "--ff-only", "-q"], check=True)
    target = f".[{EXTRAS}]" if EXTRAS else "."
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", target], check=True)

    # The running kernel doesn't pick up an editable install made after it started.
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")

    # Colab's IPython 7.34 still imports `imp`, which Python 3.13 removed.
    sys.modules.setdefault("imp", types.SimpleNamespace(reload=importlib.reload))
    shell.run_line_magic("load_ext", "autoreload")
    shell.run_line_magic("autoreload", "2")

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import polars as pl
from IPython.display import HTML, display

from movierec import paths
from movierec.evaluation import evaluate_validation

DATA_DIR = paths.data_dir()
RESULTS_DIR = paths.results_dir()
K = 10
MIN_RATINGS = 10
RELEVANT_RATING = 4.0

inputs = {
    "ratings": DATA_DIR / "ratings.parquet",
    "users": DATA_DIR / "users_sample.parquet",
    "movies": DATA_DIR / "movies.parquet",
    "recommendations": RESULTS_DIR / "popularity_val.parquet",
}
missing = [str(path) for path in inputs.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Run 01_data and 02_popularity (val) first. Missing: " + ", ".join(missing)
    )

per_user, summary = evaluate_validation(
    pl.scan_parquet(inputs["ratings"]),
    pl.read_parquet(inputs["users"]).sort("sample_rank"),
    pl.read_parquet(inputs["movies"]),
    pl.read_parquet(inputs["recommendations"]),
    k=K,
    min_ratings=MIN_RATINGS,
    relevant_rating=RELEVANT_RATING,
)
summary = summary.with_columns(pl.lit("popularity").alias("model"), pl.lit("val").alias("split"))

## Popularity results

The table reports user means and global catalog coverage. The chart shows the metrics on
a 0–1 scale. Novelty is measured in bits and appears in the table.

In [ ]:
result = summary.row(0, named=True)
print(f"Accuracy cohort: {result['n_accuracy_users']:,}/{result['n_users']:,} users")
print(f"Excluded from accuracy (no validation positives): {result['n_excluded_no_positives']:,}")
print(f"Users with empty rankings: {result['n_empty_rankings']:,}")
print(f"Eligible train catalog: {result['catalog_size']:,} films")

labels = {
    "recall": f"Recall@{K}",
    "ndcg": f"NDCG@{K}",
    "novelty": f"Novelty@{K} (bits)",
    "genre_diversity": f"Genre diversity@{K}",
    "long_tail_share": f"Long-tail share@{K}",
    "catalog_coverage": "Catalog coverage",
}
display(
    pl.DataFrame(
        {
            "Metric": list(labels.values()),
            "Popularity (validation)": [result[key] for key in labels],
        },
        schema_overrides={"Popularity (validation)": pl.Float64},
    )
)

bars = []
for key, label in labels.items():
    value = result[key]
    if key == "novelty" or value is None:
        continue
    bars.append(
        f'<div style="display:flex;align-items:center;gap:8px;margin:6px 0">'
        f'<span style="width:155px">{label}</span>'
        f'<progress max="1" value="{value}" style="width:240px"></progress>'
        f"<span>{value:.4f}</span></div>"
    )
display(HTML("".join(bars)))

In [ ]:
for name, frame in {
    "popularity_val_user_metrics.parquet": per_user,
    "popularity_val_summary.parquet": summary,
}.items():
    output = RESULTS_DIR / name
    with TemporaryDirectory(dir=RESULTS_DIR) as temp_dir:
        temporary_output = Path(temp_dir) / name
        frame.write_parquet(temporary_output)
        temporary_output.replace(output)
    print(f"Saved {output}")